# 📊 Portafolio de Inversión para el Inversor Joven Conservador

**Curso:** Manejo de Datos  
**Enfoque:** Análisis cuantitativo de un portafolio de largo plazo (10–20 años)

---

## 🎯 Filosofía de inversión: ¿Por qué ser conservador siendo joven?

Existe una idea popular que dice *"eres joven, puedes asumir más riesgo"*. Eso es parcialmente cierto — tienes tiempo para recuperarte de caídas. Sin embargo, un **joven conservador** prioriza la consistencia sobre los rendimientos explosivos, por razones muy racionales:

1. **El interés compuesto favorece la paciencia.** Un rendimiento del 9% anual durante 20 años multiplica tu capital por **5.6x**. No necesitas apuestas agresivas.
2. **Las pérdidas grandes tardan más en recuperarse.** Una caída del 50% requiere un +100% para volver al punto de partida.
3. **La volatilidad afecta el comportamiento.** Los inversores que ven caer su portafolio un 40% suelen vender en el peor momento (pánico).

### 📐 Perfil del inversor que modelamos:
- 🎓 25–35 años, inicio de vida profesional
- 🎯 Horizonte de inversión: 15–20 años
- 💡 Objetivo: Crecimiento patrimonial con mínima intervención (estrategia *buy & hold*)
- 🛡️ Tolerancia al riesgo: Baja-Media — acepta volatilidad moderada, evita activos especulativos

---

## 🏗️ Construcción del Portafolio

### ¿Por qué ETFs y no acciones individuales?

Un **ETF (Exchange-Traded Fund)** es una canasta de activos que cotiza en bolsa como si fuera una sola acción. Para un inversor conservador son ideales porque:
- 📦 **Diversificación inmediata** → SPY contiene las 500 empresas más grandes de EE.UU.
- 💸 **Costos bajos** → comisiones de 0.03%–0.20% vs fondos activos que cobran 1%–2%
- 🔍 **Transparencia** → sabes exactamente qué tienes en todo momento
- 📈 **Track record probado** → décadas de datos históricos

### Los activos seleccionados y su justificación:

| Activo | Ticker | Peso | Clase | Justificación |
|--------|--------|------|-------|---------------|
| SPDR S&P 500 ETF | `SPY` | 40% | Renta Variable EE.UU. | Núcleo del portafolio. Exposición a las 500 empresas más grandes del mundo. Rendimiento histórico ~10% anual |
| Invesco QQQ (NASDAQ 100) | `QQQ` | 20% | Renta Variable Tecnología | Crecimiento en tecnología e innovación. Alto potencial a largo plazo con más volatilidad controlada |
| iShares MSCI World ETF | `URTH` | 15% | Renta Variable Global | Diversificación geográfica. Reduce dependencia de EE.UU. (Europa, Japón, mercados emergentes) |
| iShares Core US Aggregate Bond | `AGG` | 15% | Renta Fija | Bono diversificado de EE.UU. Ancla de estabilidad, bajo riesgo, correlación negativa con acciones |
| SPDR Gold Shares | `GLD` | 10% | Materia Prima | Oro como cobertura contra inflación y crisis. Se comporta bien cuando las acciones caen |

> **Nota metodológica:** Los pesos fueron definidos siguiendo el principio de la **Frontera Eficiente de Markowitz** — maximizar rendimiento esperado para un nivel de riesgo dado. Un portafolio 70% renta variable / 15% renta fija / 10% materias primas es clásico para perfil conservador-moderado.

---
## 🔧 Configuración e Instalación

In [1]:
# Instalar dependencias (solo primera vez)
# keras y tensorflow son necesarios para la sección de redes neuronales
!pip install yfinance plotly ipywidgets keras tensorflow scipy --quiet

# Activar widgets en Jupyter/Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Google Colab detectado — widgets activados")
except ImportError:
    print("✅ Jupyter local detectado")

print("✅ Instalación completa")

✅ Jupyter local detectado
✅ Instalación completa


In [2]:
# ─── Importaciones ────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta, date
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


---
## 📥 Sección 1: Descarga de Datos

Define el portafolio, selecciona el período y descarga los precios históricos desde Yahoo Finance.  
Presiona el botón para iniciar los datos quedan guardados en memoria para todas las secciones siguientes.

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL PORTAFOLIO (constantes globales)
# ════════════════════════════════════════════════════════════

# TODO 1: Define el diccionario PORTAFOLIO con los 5 activos.
# Cada activo debe tener: 'peso' (float), 'nombre' (str), 'color' (hex str), 'clase' (str)
# Activos: SPY (40%), QQQ (20%), URTH (15%), AGG (15%), GLD (10%)
PORTAFOLIO = {
    # TU CÓDIGO AQUÍ
}

TICKERS      = list(PORTAFOLIO.keys())
PESOS        = np.array([PORTAFOLIO[t]['peso'] for t in TICKERS])
DIAS_TRADING = 252    # Días hábiles bursátiles en un año
TASA_RF      = 0.045  # Tasa libre de riesgo (~T-Bills 2024)

# Constantes dadas — no modificar
DIAS_TRADING = 252    # Días hábiles bursátiles en un año
TASA_RF      = 0.045  # Tasa libre de riesgo (~T-Bills 2024)

# Variables globales — se llenarán al presionar el botón
PRECIOS         = None
RENDIMIENTOS    = None
REND_PORTAFOLIO = None

print('Portafolio configurado:')
for t, info in PORTAFOLIO.items():
    print(f'   {t}: {info["nombre"]} — {int(info["peso"]*100)}%')


📋 Portafolio configurado:
   SPY: S&P 500 ETF — 40%
   QQQ: NASDAQ 100 ETF — 20%
   URTH: MSCI World ETF — 15%
   AGG: US Bonds ETF — 15%
   GLD: Gold ETF — 10%


In [ ]:
# ─── Widgets de configuración (provistos — no modificar) ─────────────────────
start_date_picker = widgets.DatePicker(
    description='Fecha de Inicio:', value=date(2005, 1, 1),
    disabled=False, style={'description_width': 'initial'}
)
end_date_picker = widgets.DatePicker(
    description='Fecha de Fin:', value=date.today(),
    disabled=False, style={'description_width': 'initial'}
)
descarga_output = widgets.Output()


def descargar_datos(b=None):
    """
    Descarga precios históricos de Yahoo Finance, calcula rendimientos
    logarítmicos y el rendimiento ponderado del portafolio.
    Completa los TODO marcados con # TU CÓDIGO AQUÍ
    """
    global PRECIOS, RENDIMIENTOS, REND_PORTAFOLIO

    with descarga_output:
        descarga_output.clear_output()
        print(f'⏳ Descargando datos para: {TICKERS}')
        print(f'   Período: {start_date_picker.value} → {end_date_picker.value}\n')

        try:
            # TODO 3: Descarga los datos históricos con yfinance.
            # Usa: tickers=TICKERS, start, end desde los pickers, auto_adjust=True, progress=False
            raw = # TU CÓDIGO AQUÍ

            # TODO 4: Extrae solo los precios de cierre ('Close').
            # yfinance puede devolver MultiIndex — maneja ambos casos.
            # Aplica .ffill().dropna() al final.
            if isinstance(raw.columns, pd.MultiIndex):
                PRECIOS = # TU CÓDIGO AQUÍ
            else:
                PRECIOS = # TU CÓDIGO AQUÍ

            # Rendimientos logarítmicos diarios: log(P_t / P_{t-1})
            # Son preferidos en finanzas porque son aditivos en el tiempo
            RENDIMIENTOS =  np.log(PRECIOS / PRECIOS.shift(1)).dropna()

            # # Rendimiento del portafolio = suma ponderada de rendimientos individuales
            REND_PORTAFOLIO = (RENDIMIENTOS * PESOS).sum(axis=1)

            print(f'✅ Datos descargados correctamente:')
            print(f'   → {len(PRECIOS):,} días de trading')
            print(f'   → Período: {PRECIOS.index[0].date()} → {PRECIOS.index[-1].date()}')
            print(f'   → Activos: {list(PRECIOS.columns)}')

            # TODO 7: Construye la tabla de métricas (DataFrame con índice TICKERS).
            # Columnas requeridas: 'Nombre', 'Peso (%)', 'Rend. Anual (%)', 'Volatilidad (%)',
            #                      'Sharpe Ratio', 'Rend. Total (%)'
            # Pistas:
            #   Rend. Anual  = media diaria × 252 × 100
            #   Volatilidad  = std diaria × sqrt(252) × 100
            #   Sharpe       = (Rend_anual/100 - TASA_RF) / (Vol/100)
            #   Rend. Total  = (precio_final / precio_inicial - 1) × 100
            metricas = pd.DataFrame(index=TICKERS)
            # TU CÓDIGO AQUÍ

            # TODO 8: Calcula rend. anual, volatilidad y Sharpe del portafolio combinado
            # usando REND_PORTAFOLIO (mismas fórmulas que arriba)
            rend_pa  = # TU CÓDIGO AQUÍ
            vol_pa   = # TU CÓDIGO AQUÍ
            sharpe_p = # TU CÓDIGO AQUÍ

            print('\n📊 Métricas del portafolio (período completo):')
            display(metricas)
            print(f'\n🏆 Portafolio combinado:')
            print(f'   Rendimiento anual:  {rend_pa:.2f}%')
            print(f'   Volatilidad anual:  {vol_pa:.2f}%')
            print(f'   Sharpe Ratio:       {sharpe_p:.3f}')

            # TODO 9: Grafica el rendimiento acumulado base 100.
            # - Para cada activo: normaliza su precio dividiéndolo por el primero × 100
            # - Para el portafolio: usa (1 + REND_PORTAFOLIO).cumprod() × 100
            # - El portafolio debe ir en negro, línea discontinua (linestyle='--'), linewidth=2.5
            # - Agrega título, ejes, leyenda y grid
            valor_port = # TU CÓDIGO AQUÍ

            fig, ax = plt.subplots(figsize=(11, 4))
            # TU CÓDIGO AQUÍ

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f'❌ Error al descargar datos: {e}')


# Botón — no modificar
button_descarga = widgets.Button(
    description='📥 Descargar Datos del Portafolio',
    button_style='primary',
    layout=Layout(width='300px', height='38px')
)
button_descarga.on_click(descargar_datos)
display(HBox([start_date_picker, end_date_picker]))
display(button_descarga, descarga_output)


## Resultado esperado: 
![imagen](https://raw.githubusercontent.com/jugernaut/ManejoDatos/main/Imagenes/proyecto_piloto_cartera_inversion/Img_2.png)
![imagen](https://raw.githubusercontent.com/jugernaut/ManejoDatos/main/Imagenes/proyecto_piloto_cartera_inversion/Img_3.png)

---
## 📈 Sección 2: Dashboard Interactivo

Selecciona el período y tipo de visualización, luego presiona el botón para actualizar la gráfica.

In [1]:
# ════════════════════════════════════════════════════════════
# DASHBOARD — Selección de período y tipo de gráfica
# ════════════════════════════════════════════════════════════

# Widgets provistos — no modificar
PERIODOS = {
    '1 Semana': 7, '1 Mes': 30, '3 Meses': 90, '6 Meses': 180,
    '1 Año': 365, '3 Años': 365*3, '5 Años': 365*5, '10 Años': 365*10, 'Máximo': 365*20
}
selector_periodo = widgets.ToggleButtons(
    options=list(PERIODOS.keys()), value='5 Años',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=Layout(width='100%')
)
selector_tipo = widgets.RadioButtons(
    options=[
        ('Rendimiento acumulado (Base 100)', 'norm'),
        ('Rendimientos diarios (%)', 'rend'),
        ('Volatilidad móvil 30 días', 'vol'),
        ('Drawdown (caída desde máximo)', 'dd'),
    ],
    value='norm', style={'description_width': 'initial'}, layout=Layout(width='370px')
)
checks_activos = [
    widgets.Checkbox(
        value=True,
        description=f"{t} — {PORTAFOLIO[t]['nombre']} ({int(PORTAFOLIO[t]['peso']*100)}%)",
        style={'description_width': 'initial'}, layout=Layout(width='400px')
    ) for t in TICKERS
]
check_portafolio = widgets.Checkbox(
    value=True, description='📊 Mostrar portafolio ponderado',
    style={'description_width': 'initial'}, layout=Layout(width='300px')
)
dashboard_output = widgets.Output()


def actualizar_dashboard(b=None):
    with dashboard_output:
        dashboard_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        activos_sel  = [t for t, cb in zip(TICKERS, checks_activos) if cb.value]
        mostrar_port = check_portafolio.value
        dias_max     = PERIODOS[selector_periodo.value]
        tipo_graf    = selector_tipo.value

        if not activos_sel and not mostrar_port:
            print('⚠️ Selecciona al menos un activo.')
            return

        # TODO 10: Filtra los datos al período seleccionado.
        # fecha_corte = última fecha - timedelta(days=dias_max)
        # Filtra PRECIOS, RENDIMIENTOS y REND_PORTAFOLIO desde esa fecha
        fecha_corte = # TU CÓDIGO AQUÍ
        precios_f   = # TU CÓDIGO AQUÍ
        rend_f      = # TU CÓDIGO AQUÍ
        rend_port_f = # TU CÓDIGO AQUÍ

        fig, ax = plt.subplots(figsize=(12, 5))

        # TODO 11: Implementa los 4 tipos de gráfica según tipo_graf:
        #
        # 'norm' → Rendimiento acumulado base 100 (igual que Sección 1)
        #
        # 'rend' → Barras de rendimientos diarios (%) por activo
        #          Si mostrar_port: línea del portafolio encima
        #
        # 'vol'  → Volatilidad anualizada con ventana móvil de 30 días
        #          Fórmula: rolling(30).std() × sqrt(252) × 100
        #
        # 'dd'   → Drawdown: (precio - precio_máximo_hasta_hoy) / precio_máximo × 100
        #          Usa fill_between para rellenar el área bajo cero
        #          Pista: precio_máximo_hasta_hoy = s.cummax()
        if tipo_graf == 'norm':
            titulo, ylabel = f'Rendimiento Acumulado (Base 100) — {selector_periodo.value}', 'Valor indexado'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'rend':
            titulo, ylabel = f'Rendimientos Diarios (%) — {selector_periodo.value}', 'Rendimiento (%)'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'vol':
            titulo, ylabel = f'Volatilidad Anualizada 30 días (%) — {selector_periodo.value}', 'Volatilidad (%)'
            # TU CÓDIGO AQUÍ

        elif tipo_graf == 'dd':
            titulo, ylabel = f'Drawdown desde Máximo (%) — {selector_periodo.value}', 'Drawdown (%)'
            # TU CÓDIGO AQUÍ

        ax.set_title(titulo, fontsize=13, fontweight='bold')
        ax.set_xlabel('Fecha')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # TODO 12: Muestra la tabla de métricas del período seleccionado.
        # Columnas: 'Nombre', 'Rend. Anual (%)', 'Volatilidad (%)', 'Sharpe', 'Rend. Total (%)'
        if activos_sel and not rend_f[activos_sel].empty:
            m = pd.DataFrame(index=activos_sel)
            # TU CÓDIGO AQUÍ
            print(f'\n📊 Métricas del período seleccionado ({selector_periodo.value}):')
            display(m)


# Layout — no modificar
button_dashboard = widgets.Button(
    description='🔄 Actualizar Gráfica', button_style='info',
    layout=Layout(width='200px', height='38px')
)
button_dashboard.on_click(actualizar_dashboard)
panel_checks = VBox(
    [widgets.HTML('<b>📦 Activos</b>')] + checks_activos + [check_portafolio],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='430px')
)
panel_tipo = VBox(
    [widgets.HTML('<b>📉 Visualización</b>'), selector_tipo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='380px')
)
panel_periodo = VBox(
    [widgets.HTML('<b>📅 Período</b>'), selector_periodo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px')
)
display(HBox([panel_checks, panel_tipo], layout=Layout(gap='12px', margin='0 0 10px 0')))
display(panel_periodo)
display(button_dashboard)
display(dashboard_output)


SyntaxError: invalid syntax (2932869428.py, line 58)

## Resultado esperado
![imagen](https://raw.githubusercontent.com/jugernaut/ManejoDatos/main/Imagenes/proyecto_piloto_cartera_inversion/Img_4.png)
![imagen](https://raw.githubusercontent.com/jugernaut/ManejoDatos/main/Imagenes/proyecto_piloto_cartera_inversion/Img_5.png)

---
## 📐 Sección 3: Análisis Cuantitativo — Correlaciones y Simulación histórica

In [6]:
# ─── Análisis cuantitativo: correlación + simulación histórica ────────────────

analisis_output = widgets.Output()


def analisis_cuantitativo(b=None):
    with analisis_output:
        analisis_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        # TODO 13: Calcula la matriz de correlación entre los rendimientos.
        # Usa .corr().round(3) sobre RENDIMIENTOS
        corr = # TU CÓDIGO AQUÍ
        nombres = [PORTAFOLIO[t]['nombre'] for t in TICKERS]

        # TODO 14: Grafica la matriz de correlación como heatmap.
        # - Usa ax.imshow con cmap='RdYlGn', vmin=-1, vmax=1
        # - Agrega colorbar, etiquetas en ejes x e y (usa nombres)
        # - Muestra el valor numérico en cada celda con ax.text()
        fig, ax = plt.subplots(figsize=(7, 5))
        # TU CÓDIGO AQUÍ

        ax.set_title('Correlación entre Rendimientos del Portafolio', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print('💡 El oro (GLD) y los bonos (AGG) tienen correlación baja/negativa con acciones → buena diversificación.')

        # TODO 15: Simulación histórica — ¿cuánto valdría $10,000 invertidos hace 10 años?
        # 1. Filtra precios y rendimiento del portafolio a los últimos 10 años
        # 2. Calcula el valor acumulado: INVERSION × (1 + rend_port_10y).cumprod()
        # 3. Grafica el valor de cada activo individual (100% en ese activo) y el portafolio
        # 4. Imprime: inversión inicial, valor final, ganancia y múltiplo
        INVERSION = 10_000
        # TU CÓDIGO AQUÍ


button_analisis = widgets.Button(
    description='📊 Mostrar Correlaciones y Simulación Histórica',
    button_style='success', layout=Layout(width='380px', height='38px')
)
button_analisis.on_click(analisis_cuantitativo)
display(button_analisis, analisis_output)


Button(button_style='success', description='📊 Mostrar Correlaciones y Simulación Histórica', layout=Layout(hei…

Output()

![imagen](https://raw.githubusercontent.com/jugernaut/ManejoDatos/main/Imagenes/proyecto_piloto_cartera_inversion/Img_6.png)

---
## 💼 Sección 4: Proyección de tu Inversión — Monte Carlo vs Red Neuronal LSTM

En esta sección tú decides **cuánto dinero quieres invertir** en el portafolio completo y comparamos dos métodos de proyección para estimar cuánto podrías tener en el futuro:

1. **Simulación de Monte Carlo** — genera miles de escenarios probabilísticos basados en el comportamiento histórico del portafolio.
2. **Red Neuronal LSTM** — aprende patrones en los datos históricos del SPY para proyectar una trayectoria futura.

Al final verás una **tabla comparativa** entre ambos métodos y una **gráfica de ganancias proyectadas** según el monto que decidas invertir.

---

### 🎲 ¿Qué es la Simulación de Monte Carlo?

La **simulación de Monte Carlo** es una técnica matemática que usa **números aleatorios** para modelar situaciones con incertidumbre. En finanzas se usa para responder: *¿cómo podría evolucionar el valor de mi inversión en el futuro?*

**Idea central:** Si conocemos el rendimiento promedio y la volatilidad histórica del portafolio, podemos simular miles de posibles "futuros" respetando esas estadísticas. El resultado es una **distribución de probabilidad** del valor futuro de la inversión.

#### El modelo: Movimiento Browniano Geométrico (GBM)

$$S_{t+1} = S_t \cdot \exp\left[(\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} \cdot Z\right]$$

Donde:
- $\mu$ = rendimiento promedio diario estimado históricamente
- $\sigma$ = volatilidad diaria estimada históricamente
- $Z \sim \mathcal{N}(0,1)$ = número aleatorio normal estándar
- El término $-\sigma^2/2$ es la **corrección de Jensen** que evita sesgo al usar logaritmos

**Ventajas de Monte Carlo:**
- ✅ Genera una distribución completa de escenarios con probabilidades
- ✅ Es honesto sobre la incertidumbre — la banda se ensancha con el tiempo
- ✅ Ideal para planificación financiera de largo plazo
- ✅ Alta interpretabilidad — µ y σ tienen significado claro

**Limitaciones:**
- ❌ Asume que los rendimientos siguen una distribución normal
- ❌ No captura tendencias ni patrones del mercado
- ❌ Los parámetros históricos pueden no reflejar el futuro

---

### 🤖 ¿Qué es una Red Neuronal LSTM?

Una **LSTM (Long Short-Term Memory)** es un tipo de red neuronal recurrente diseñada específicamente para series de tiempo. A diferencia de una red neuronal simple, la LSTM tiene **"memoria"**: puede recordar patrones de hace varios pasos en el tiempo y decidir qué información conservar o descartar.

#### Arquitectura de nuestra LSTM:

```
Entrada (60 días previos)
       ↓
LSTM(64 unidades, return_sequences=True)
       ↓
Dropout(0.20) — evita sobreajuste
       ↓
LSTM(32 unidades)
       ↓
Dropout(0.20)
       ↓
Dense(16, activación ReLU)
       ↓
Dense(1) — precio proyectado
```

**Ventajas de LSTM:**
- ✅ Captura tendencias y patrones recientes en los datos
- ✅ Puede detectar comportamientos no lineales
- ✅ Aprende automáticamente de los datos históricos

**Limitaciones:**
- ❌ Produce una sola trayectoria — no modela incertidumbre explícitamente
- ❌ Acumula errores en proyecciones largas
- ❌ "Caja negra" — difícil de interpretar
- ❌ Los mercados tienen aleatoriedad irreducible que ningún modelo puede capturar

---

### 📊 Tabla Comparativa: Monte Carlo vs LSTM

| Característica | Monte Carlo (GBM) | Red Neuronal LSTM |
|---|---|---|
| **Tipo de modelo** | Estadístico / Probabilístico | Machine Learning / Determinístico |
| **Supuesto clave** | Rendimientos son ruido normal | El mercado tiene patrones aprendibles |
| **Salida** | Distribución (miles de escenarios) | Una sola trayectoria |
| **Interpretabilidad** | 🟢 Alta — µ y σ tienen significado | 🔴 Baja — "caja negra" |
| **Captura tendencias** | 🔴 No | 🟢 Sí (parcialmente) |
| **Modela incertidumbre** | 🟢 Sí — con bandas de confianza | 🔴 No explícitamente |
| **Velocidad** | 🟢 Muy rápida | 🟡 Lenta (entrenamiento 1–3 min) |
| **Uso recomendado** | Planificación largo plazo | Análisis corto plazo / tendencias |
| **Riesgo de sobreajuste** | 🟢 Bajo | 🔴 Alto en series largas |
| **Datos requeridos** | Solo µ y σ históricos | Secuencias largas de precios |

> ⚠️ **Nota importante:** Ningún modelo puede predecir el futuro con certeza. Monte Carlo lo admite abiertamente mostrando un rango de posibilidades. Para un inversor de largo plazo, esa honestidad sobre la incertidumbre es más valiosa que la aparente precisión de la red neuronal.


In [ ]:
# ════════════════════════════════════════════════════════════
# SECCIÓN 4 — PROYECCIÓN: MONTE CARLO vs LSTM
# Portafolio completo con inversión personalizada
# ════════════════════════════════════════════════════════════
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import keras
from keras import layers, models
from sklearn.preprocessing import MinMaxScaler

# ── Widgets ──────────────────────────────────────────────────
inversion_widget = widgets.BoundedIntText(
    value=10_000, min=100, max=1_000_000, step=500,
    description='💵 Inversión inicial (USD):',
    style={'description_width': 'initial'},
    layout=Layout(width='380px')
)
horizonte_widget = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description='📅 Horizonte (años):',
    style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
n_sim_widget = widgets.IntSlider(
    value=1000, min=200, max=3000, step=200,
    description='🎲 N simulaciones MC:',
    style={'description_width': 'initial'},
    continuous_update=False, layout=Layout(width='420px')
)
sec4_output = widgets.Output()


def ejecutar_proyeccion(b=None):
    with sec4_output:
        sec4_output.clear_output()

        if PRECIOS is None:
            print('❌ Primero descarga los datos con el botón de la Sección 1.')
            return

        INV     = inversion_widget.value
        H_ANIOS = horizonte_widget.value
        N       = n_sim_widget.value
        N_DIAS  = H_ANIOS * DIAS_TRADING
        ACTIVO  = 'SPY'
        VENTANA = 60

        print(f'💵 Inversión inicial: ${INV:,.0f}')
        print(f'📅 Horizonte: {H_ANIOS} años ({N_DIAS:,} días de trading)')
        print(f'🎲 Simulaciones Monte Carlo: {N:,}')
        print('─' * 55)

        # ── MONTE CARLO — PORTAFOLIO COMPLETO ────────────────
        print('\n⏳ Ejecutando Monte Carlo sobre el portafolio completo...')
        np.random.seed(42)

        ultimos_5y = PRECIOS.index[-1] - timedelta(days=5 * 365)
        rend_rec   = RENDIMIENTOS.loc[RENDIMIENTOS.index >= ultimos_5y]
        medias     = rend_rec.mean().values
        cov_d      = rend_rec.cov().values
        varianzas  = np.diag(cov_d)
        L          = np.linalg.cholesky(cov_d)
        drift_vec  = medias - 0.5 * varianzas

        valores_mc = np.zeros((N_DIAS + 1, N))
        valores_mc[0, :] = INV
        for sim in range(N):
            Z        = np.random.standard_normal((5, N_DIAS))
            shocks   = L @ Z
            rend_act = drift_vec.reshape(-1, 1) + shocks
            rend_p   = PESOS @ rend_act
            valores_mc[1:, sim] = INV * np.cumprod(np.exp(rend_p))

        p5_mc  = np.percentile(valores_mc, 5,  axis=1)
        p25_mc = np.percentile(valores_mc, 25, axis=1)
        p50_mc = np.percentile(valores_mc, 50, axis=1)
        p75_mc = np.percentile(valores_mc, 75, axis=1)
        p95_mc = np.percentile(valores_mc, 95, axis=1)

        ganancia_mediana_mc = p50_mc[-1] - INV
        multiplo_mc         = p50_mc[-1] / INV
        print(f'✅ Monte Carlo completo')
        print(f'   Valor mediano final: ${p50_mc[-1]:,.0f}')
        print(f'   Ganancia mediana:    ${ganancia_mediana_mc:,.0f}')
        print(f'   Múltiplo:            {multiplo_mc:.2f}x')

        # ── LSTM — SPY ────────────────────────────────────────
        print(f'\n⏳ Entrenando LSTM para proyectar SPY ({H_ANIOS} años)...')
        print('   (puede tardar 1–3 minutos en CPU)')

        precios_spy = PRECIOS[[ACTIVO]].copy()
        scaler      = MinMaxScaler()
        precios_esc = scaler.fit_transform(precios_spy)

        split      = int(len(precios_esc) * 0.80)
        train_data = precios_esc[:split]
        test_data  = precios_esc[split:]

        def crear_seq(datos, ventana):
            X, y = [], []
            for i in range(ventana, len(datos)):
                X.append(datos[i - ventana:i, 0])
                y.append(datos[i, 0])
            return np.array(X), np.array(y)

        X_train, y_train = crear_seq(train_data, VENTANA)
        X_test,  y_test  = crear_seq(
            np.concatenate([train_data[-VENTANA:], test_data]), VENTANA
        )
        X_train = X_train.reshape(-1, VENTANA, 1)
        X_test  = X_test.reshape(-1, VENTANA, 1)

        keras.utils.set_random_seed(42)
        modelo = models.Sequential([
            layers.LSTM(64, return_sequences=True, input_shape=(VENTANA, 1)),
            layers.Dropout(0.20),
            layers.LSTM(32, return_sequences=False),
            layers.Dropout(0.20),
            layers.Dense(16, activation='relu'),
            layers.Dense(1)
        ], name='LSTM_SPY')
        modelo.compile(optimizer='adam', loss='mean_squared_error')
        hist = modelo.fit(
            X_train, y_train,
            epochs=30, batch_size=32,
            validation_split=0.10, verbose=0
        )
        print(f'✅ LSTM entrenada — Loss final: {hist.history["loss"][-1]:.6f}')

        # Proyección futura con ventana deslizante
        secuencia = precios_esc[-VENTANA:].reshape(1, VENTANA, 1).copy()
        pred_esc  = []
        for _ in range(N_DIAS):
            siguiente = modelo.predict(secuencia, verbose=0)[0, 0]
            pred_esc.append(siguiente)
            secuencia = np.roll(secuencia, -1, axis=1)
            secuencia[0, -1, 0] = siguiente

        pred_lstm = scaler.inverse_transform(
            np.array(pred_esc).reshape(-1, 1)
        ).flatten()

        # Convertir trayectoria de precios SPY → valor del portafolio
        # Asumimos que la fracción de SPY en el portafolio sigue la proyección LSTM
        # y el resto crece a su rendimiento histórico promedio anualizado
        spy_weight  = PESOS[TICKERS.index(ACTIVO)]
        rend_otros  = float((rend_rec.drop(columns=ACTIVO) * PESOS[
            [i for i, t in enumerate(TICKERS) if t != ACTIVO]
        ]).sum(axis=1).mean() * DIAS_TRADING)

        S0_spy      = float(precios_spy[ACTIVO].iloc[-1])
        factor_spy  = pred_lstm / S0_spy
        factor_otros = np.exp(rend_otros * np.linspace(0, H_ANIOS, N_DIAS))
        valor_lstm  = INV * (spy_weight * factor_spy + (1 - spy_weight) * factor_otros)
        valor_lstm  = np.concatenate([[INV], valor_lstm])

        ganancia_lstm = valor_lstm[-1] - INV
        multiplo_lstm = valor_lstm[-1] / INV
        print(f'\n✅ Proyección LSTM completa')
        print(f'   Valor final proyectado: ${valor_lstm[-1]:,.0f}')
        print(f'   Ganancia proyectada:    ${ganancia_lstm:,.0f}')
        print(f'   Múltiplo:               {multiplo_lstm:.2f}x')

        dias_eje = np.arange(N_DIAS + 1)
        anios_eje = dias_eje / DIAS_TRADING

        # ── GRÁFICA 1: Proyección comparativa ────────────────
        fig, ax = plt.subplots(figsize=(13, 6))

        # Monte Carlo — bandas
        ax.fill_between(anios_eje, p5_mc,  p95_mc,
                        alpha=0.12, color='steelblue', label='MC: Banda 90% (P5–P95)')
        ax.fill_between(anios_eje, p25_mc, p75_mc,
                        alpha=0.22, color='steelblue', label='MC: Banda 50% (P25–P75)')
        ax.plot(anios_eje, p50_mc,
                color='steelblue', lw=2.5, label=f'MC: Mediana → ${p50_mc[-1]:,.0f}')
        ax.plot(anios_eje, p95_mc,
                color='steelblue', lw=1, ls='--', alpha=0.6, label=f'MC: P95 → ${p95_mc[-1]:,.0f}')
        ax.plot(anios_eje, p5_mc,
                color='steelblue', lw=1, ls=':', alpha=0.6, label=f'MC: P5  → ${p5_mc[-1]:,.0f}')

        # LSTM
        ax.plot(anios_eje, valor_lstm,
                color='tomato', lw=2.5, label=f'LSTM → ${valor_lstm[-1]:,.0f}')

        # Inversión inicial
        ax.axhline(INV, color='gray', lw=1.2, ls='--', alpha=0.7, label=f'Inversión inicial: ${INV:,.0f}')

        ax.set_title(
            f'Proyección de ${INV:,.0f} invertidos — {H_ANIOS} años\nMonte Carlo vs Red Neuronal LSTM',
            fontsize=13, fontweight='bold'
        )
        ax.set_xlabel('Años desde hoy', fontsize=11)
        ax.set_ylabel('Valor del portafolio (USD)', fontsize=11)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.legend(fontsize=9, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── GRÁFICA 2: Ganancia proyectada ───────────────────
        fig2, axes = plt.subplots(1, 2, figsize=(13, 5))

        # Panel izquierdo — barras de ganancia MC por percentil
        labels_perc = ['P5\n(pesimista)', 'P25', 'Mediana\n(P50)', 'P75', 'P95\n(optimista)', 'LSTM']
        ganancias   = [
            p5_mc[-1]  - INV,
            p25_mc[-1] - INV,
            p50_mc[-1] - INV,
            p75_mc[-1] - INV,
            p95_mc[-1] - INV,
            valor_lstm[-1] - INV,
        ]
        colores = ['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c', '#337ab7', '#e74c3c']
        bars = axes[0].bar(labels_perc, ganancias, color=colores, edgecolor='white', linewidth=0.8)
        axes[0].axhline(0, color='black', lw=1)
        for bar, g in zip(bars, ganancias):
            axes[0].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(ganancias) * 0.02,
                f'${g:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold'
            )
        axes[0].set_title(f'Ganancia proyectada a {H_ANIOS} años\n(sobre ${INV:,.0f} invertidos)', fontsize=11, fontweight='bold')
        axes[0].set_ylabel('Ganancia (USD)', fontsize=10)
        axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0].grid(True, axis='y', alpha=0.3)

        # Panel derecho — múltiplos
        multiplos = [v / INV for v in [p5_mc[-1], p25_mc[-1], p50_mc[-1], p75_mc[-1], p95_mc[-1], valor_lstm[-1]]]
        bars2 = axes[1].bar(labels_perc, multiplos, color=colores, edgecolor='white', linewidth=0.8)
        axes[1].axhline(1.0, color='black', lw=1, ls='--', label='Capital inicial (1x)')
        for bar, m in zip(bars2, multiplos):
            axes[1].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(multiplos) * 0.02,
                f'{m:.2f}x', ha='center', va='bottom', fontsize=9, fontweight='bold'
            )
        axes[1].set_title(f'Múltiplo del capital a {H_ANIOS} años', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Múltiplo (veces el capital inicial)', fontsize=10)
        axes[1].legend(fontsize=9)
        axes[1].grid(True, axis='y', alpha=0.3)

        plt.suptitle(f'Resumen de proyección — ${INV:,.0f} durante {H_ANIOS} años', fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

        # ── TABLA RESUMEN ─────────────────────────────────────
        print('\n' + '═' * 55)
        print(f'📊 RESUMEN DE PROYECCIÓN — ${INV:,.0f} durante {H_ANIOS} años')
        print('═' * 55)
        resumen = pd.DataFrame({
            'Escenario': ['MC P5 (pesimista)', 'MC P25', 'MC Mediana (P50)', 'MC P75', 'MC P95 (optimista)', 'LSTM'],
            'Valor final (USD)': [f'${v:,.0f}' for v in [p5_mc[-1], p25_mc[-1], p50_mc[-1], p75_mc[-1], p95_mc[-1], valor_lstm[-1]]],
            'Ganancia (USD)': [f'${g:,.0f}' for g in ganancias],
            'Múltiplo': [f'{m:.2f}x' for m in multiplos],
        })
        display(resumen)

        # Probabilidades MC
        vals_finales = valores_mc[-1, :]
        print(f'\n📈 Probabilidades (Monte Carlo, {N:,} simulaciones):')
        print(f'   Prob. de terminar con más dinero que hoy:   {(vals_finales > INV).mean():.1%}')
        print(f'   Prob. de duplicar la inversión (2x):        {(vals_finales > INV*2).mean():.1%}')
        print(f'   Prob. de triplicar la inversión (3x):       {(vals_finales > INV*3).mean():.1%}')
        print(f'   Prob. de perder más del 25%:                {(vals_finales < INV*0.75).mean():.1%}')


button_sec4 = widgets.Button(
    description='🚀 Ejecutar Proyección (MC + LSTM)',
    button_style='danger',
    layout=Layout(width='340px', height='42px')
)
button_sec4.on_click(ejecutar_proyeccion)

display(VBox([
    widgets.HTML('<b>⚙️ Configura tu proyección:</b>'),
    inversion_widget,
    horizonte_widget,
    n_sim_widget,
    button_sec4,
    sec4_output
]))


---
## ✅ Sección 5: Conclusiones

### 💼 ¿Cuánto podría ganar con mi inversión?

Los resultados que acabas de ver muestran que el tiempo es el activo más valioso de un inversor joven. El **interés compuesto**, combinado con una cartera diversificada y disciplina para no vender en momentos de pánico, es la herramienta más poderosa disponible.

---

### 🎲 Conclusiones sobre Monte Carlo

- Genera una **distribución completa de escenarios** — no una sola respuesta, sino un rango honesto de posibilidades.
- La **banda de confianza se ensancha con el tiempo**, lo que refleja fielmente que el futuro es más incierto cuanto más lejano está.
- El escenario mediano (P50) no es una predicción, sino el punto donde la mitad de los escenarios simulados terminan por encima y la mitad por debajo.
- Para **planificación financiera de largo plazo** (10–20 años), Monte Carlo es la herramienta más apropiada y honesta.

---

### 🤖 Conclusiones sobre la Red Neuronal LSTM

- La LSTM aprende **patrones y tendencias recientes** en los datos históricos del SPY.
- Produce una **sola trayectoria determinista** — lo que puede transmitir una falsa sensación de precisión.
- A medida que se proyecta más lejos en el futuro, **los errores se acumulan** y la proyección se aleja cada vez más de la realidad.
- Es más útil para **análisis de corto plazo** o para identificar señales de momentum en el mercado.

---

### 🏆 ¿Cuál método es mejor?

**Depende del objetivo:**

| Objetivo | Método recomendado |
|---|---|
| Planificación de retiro / largo plazo | ✅ Monte Carlo |
| Cuantificar el riesgo de perder dinero | ✅ Monte Carlo |
| Detectar tendencias recientes | ✅ LSTM |
| Señales de trading de corto plazo | ✅ LSTM |
| Comunicar incertidumbre al inversor | ✅ Monte Carlo |

---

### 📌 Reflexión final

> *Ningún modelo puede predecir el futuro con certeza. Monte Carlo lo admite abiertamente mostrando un rango de posibilidades. Para un inversor de largo plazo, esa honestidad sobre la incertidumbre es más valiosa que la aparente precisión de la red neuronal.*

**Los 3 factores que más impactan el resultado de tu inversión:**
1. ⏳ **El tiempo** — empezar antes tiene más impacto que invertir más.
2. 💰 **La consistencia** — aportar regularmente (aunque sea poco) construye patrimonio.
3. 🧘 **La paciencia** — no vender en las caídas. El portafolio conservador está diseñado para soportar la volatilidad.

---

> ⚠️ **Disclaimer:** Este notebook es exclusivamente educativo. No constituye asesoramiento financiero ni de inversión. Los rendimientos pasados no garantizan resultados futuros.
